# Data Dictionary — Indian Mutual Fund Analytics Dataset

**Version:** 1.0  
**Last Updated:** 2025-12-31 (portfolio snapshot date)  
**Data Coverage:** January 2022 – May 2026  
**Source Domain:** Indian mutual fund industry (AMFI, SEBI)  
**Total Files:** 10 CSV files

---

## Table of Contents

1. [01_fund_master.csv — Fund Master Reference](#1-01_fund_mastercsv--fund-master-reference)
2. [02_nav_history.csv — Daily NAV History](#2-02_nav_historycsv--daily-nav-history)
3. [03_aum_by_fund_house.csv — AUM by Fund House](#3-03_aum_by_fund_housecsv--aum-by-fund-house)
4. [04_monthly_sip_inflows.csv — Monthly SIP Inflows](#4-04_monthly_sip_inflowscsv--monthly-sip-inflows)
5. [05_category_inflows.csv — Category-Level Inflows](#5-05_category_inflowscsv--category-level-inflows)
6. [06_industry_folio_count.csv — Industry Folio Count](#6-06_industry_folio_countcsv--industry-folio-count)
7. [07_scheme_performance.csv — Scheme Performance](#7-07_scheme_performancecsv--scheme-performance)
8. [08_investor_transactions.csv — Investor Transactions](#8-08_investor_transactionscsv--investor-transactions)
9. [09_portfolio_holdings.csv — Portfolio Holdings](#9-09_portfolio_holdingscsv--portfolio-holdings)
10. [10_benchmark_indices.csv — Benchmark Indices](#10-10_benchmark_indicescsv--benchmark-indices)
11. [Entity Relationships](#entity-relationships)
12. [Glossary](#glossary)

---

## 1. `01_fund_master.csv` — Fund Master Reference

**Description:** Static reference table for every mutual fund scheme in the dataset. One row per scheme. Acts as the primary dimension table joined to most other files via `amfi_code`.

**Row count:** 40  
**Grain:** One row per unique mutual fund scheme

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `amfi_code` | `int64` | No | Unique scheme identifier assigned by AMFI (Association of Mutual Funds in India). Primary key for joining across tables. | Positive integer; e.g. `119551` |
| `fund_house` | `string` | No | Name of the Asset Management Company (AMC) that manages the scheme. | e.g. `SBI Mutual Fund`, `HDFC Mutual Fund`, `ICICI Prudential MF` |
| `scheme_name` | `string` | No | Full display name of the mutual fund scheme including plan type (Regular/Direct) and option (Growth). | Free text; typically includes AMC prefix, sub-category, plan, and option |
| `category` | `string` | No | Broad SEBI-mandated asset class of the scheme. | `Equity`, `Debt` |
| `sub_category` | `string` | No | SEBI sub-classification within the broad category. Defines the investment mandate. | `Large Cap`, `Mid Cap`, `Small Cap`, `Large & Mid Cap`, `Flexi Cap`, `ELSS`, `Index`, `Index/ETF`, `Value`, `Gilt`, `Liquid`, `Short Duration` |
| `plan` | `string` | No | Distribution plan type. Direct plans have no distributor commission; Regular plans include commission. | `Direct`, `Regular` |
| `launch_date` | `string (date)` | No | Date when the scheme was launched and opened for investment. Format: `YYYY-MM-DD`. | Range: historically varies; some schemes pre-2000 |
| `benchmark` | `string` | No | Name of the index against which scheme performance is measured. | e.g. `NIFTY 100 TRI`, `BSE 250 SmallCap TRI`, `CRISIL Composite Bond Fund Index` |
| `expense_ratio_pct` | `float64` | No | Annual total expense ratio (TER) charged by the AMC as a percentage of AUM. Deducted daily from NAV. | Direct plans typically 0.1–1.0%; Regular plans 0.5–2.5% |
| `exit_load_pct` | `float64` | No | Percentage fee charged on redemption within the exit load period (typically 1 year for equity funds). | `0.0` (no load) or `1.0` (1%) in this dataset |
| `min_sip_amount` | `int64` | No | Minimum monthly instalment amount permitted for a Systematic Investment Plan (SIP) in Indian Rupees (₹). | Typically ₹500 or ₹1,000 |
| `min_lumpsum_amount` | `int64` | No | Minimum one-time investment amount for lumpsum purchase in Indian Rupees (₹). | Typically ₹1,000 or ₹5,000 |
| `fund_manager` | `string` | No | Name of the primary fund manager responsible for investment decisions. | Free text; e.g. `Sohini Andani`, `R. Srinivasan` |
| `risk_category` | `string` | No | SEBI-mandated risk label displayed on all scheme documents (riskometer). | `Low`, `Moderate`, `Moderately High`, `High`, `Very High` |
| `sebi_category_code` | `string` | No | Alphanumeric code assigned by SEBI to identify the scheme category for regulatory classification. | e.g. `EC01` (Equity Large Cap), `EC03` (Equity Small Cap), `D01` (Debt Liquid) |

---

## 2. `02_nav_history.csv` — Daily NAV History

**Description:** Time-series of daily Net Asset Values (NAV) for each scheme. Used for return calculations, performance attribution, and price trend analysis.

**Row count:** 46,000  
**Grain:** One row per scheme per trading date

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `amfi_code` | `int64` | No | Foreign key to `01_fund_master.amfi_code`. Identifies the scheme. | Must match a value in fund master |
| `date` | `string (date)` | No | Trading date for which the NAV is reported. AMFI publishes NAV after market close for each business day. Format: `YYYY-MM-DD`. | Range: `2022-01-03` to `2026-05-29`; excludes weekends and market holidays |
| `nav` | `float64` | No | Official Net Asset Value per unit in Indian Rupees (₹), as declared by the AMC and published by AMFI. Reflects the per-unit market value of the scheme's portfolio after all fees and expenses. | Positive decimal; values range from ~₹26 to ~₹219 in this dataset |

**Notes:**
- NAV is calculated as: (Market value of all assets − Liabilities) ÷ Total units outstanding
- For index funds and ETFs, NAV closely tracks the underlying index value
- Missing dates indicate market holidays or non-business days (no trading)

---

## 3. `03_aum_by_fund_house.csv` — AUM by Fund House

**Description:** Quarterly snapshot of Assets Under Management (AUM) for each fund house. Sourced from AMFI monthly AUM disclosures. Used for market share and industry concentration analysis.

**Row count:** 90  
**Grain:** One row per fund house per reporting date

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `date` | `string (date)` | No | Quarter-end reporting date on which AUM was measured. Format: `YYYY-MM-DD`. | Quarterly snapshots (March 31, June 30, September 30, December 31) |
| `fund_house` | `string` | No | Name of the Asset Management Company (AMC). | e.g. `SBI Mutual Fund`, `ICICI Prudential MF`, `HDFC Mutual Fund` |
| `aum_lakh_crore` | `float64` | No | Total AUM of the fund house in lakh crore Indian Rupees (₹ Lakh Crore = ₹10 trillion). Used for high-level industry size comparisons. | Positive decimal; e.g. `6.05` = ₹6.05 lakh crore |
| `aum_crore` | `int64` | No | Total AUM of the fund house in crore Indian Rupees (₹ Crore = ₹10 million). More precise integer representation of the same AUM figure. | Integer; e.g. `605000` = ₹6,05,000 crore |
| `num_schemes` | `int64` | No | Total number of active schemes managed by the fund house as of the reporting date. Includes all open-ended and close-ended schemes. | Positive integer; typically 100–300 for large AMCs |

**Notes:**
- `aum_crore` = `aum_lakh_crore` × 1,00,000 (both represent the same underlying value)
- AUM figures include equity, debt, hybrid, and other scheme categories

---

## 4. `04_monthly_sip_inflows.csv` — Monthly SIP Inflows

**Description:** Industry-level monthly aggregate data on Systematic Investment Plan (SIP) flows. Sourced from AMFI monthly SIP data releases. Key indicator of retail investor participation.

**Row count:** 48  
**Grain:** One row per calendar month (industry aggregate)

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `month` | `string (date)` | No | First day of the calendar month for which SIP data is reported. Format: `YYYY-MM`. | Range: `2022-01` to `2025-12` |
| `sip_inflow_crore` | `int64` | No | Total SIP contribution collected across all fund houses during the month, in ₹ crore. Represents systematic investments by existing SIP registrations. | Positive integer; typically ₹11,000–₹25,000 crore per month |
| `active_sip_accounts_crore` | `float64` | No | Total number of active (live) SIP mandates at month-end, in crore (10 million) units. A mandate is 'active' if it has at least one future instalment pending. | Positive decimal; e.g. `4.91` = 4.91 crore (49.1 million) accounts |
| `new_sip_accounts_lakh` | `float64` | No | Number of new SIP registrations created during the month, in lakh (100,000) units. Measures fresh retail participation. | Positive decimal; e.g. `9.1` = 9.1 lakh (910,000) new registrations |
| `sip_aum_lakh_crore` | `float64` | No | Total AUM contributed through SIP investments, as of month-end, in ₹ lakh crore. Represents the cumulative market value of assets held by SIP investors. | Positive decimal |
| `yoy_growth_pct` | `float64` | **Yes** | Year-over-year percentage growth in `sip_inflow_crore` compared to the same month in the prior year. Null for the first 12 months (January–December 2022) due to no prior-year baseline. | Float; can be positive or negative; 12 nulls for 2022 base period |

---

## 5. `05_category_inflows.csv` — Category-Level Inflows

**Description:** Monthly net inflow data segmented by fund category. Captures investor flows (purchases minus redemptions) into different scheme types. Sourced from AMFI monthly data.

**Row count:** 144  
**Grain:** One row per fund category per month

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `month` | `string (date)` | No | Calendar month of the inflow data. Format: `YYYY-MM`. | Range: `2024-04` to `2025-12` (12 months × 12 categories) |
| `category` | `string` | No | Fund sub-category for which net inflow is reported. | `Large Cap`, `Mid Cap`, `Small Cap`, `Large & Mid Cap`, `Flexi Cap`, `ELSS`, `Index`, `Value`, `Gilt`, `Liquid`, `Short Duration`, `Hybrid` |
| `net_inflow_crore` | `float64` | No | Net inflow into the category during the month in ₹ crore. Calculated as total subscriptions minus total redemptions. A negative value indicates net outflow. | Decimal; can be negative (net outflow) |

---

## 6. `06_industry_folio_count.csv` — Industry Folio Count

**Description:** Quarterly industry-level counts of investor folios (accounts) broken down by scheme category. A folio is a unique investor account with an AMC, and investors may hold multiple folios. Sourced from AMFI quarterly reports.

**Row count:** 21  
**Grain:** One row per quarter

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `month` | `string (date)` | No | Quarter-end month for which folio data is reported. Format: `YYYY-MM`. | Quarterly; `2022-01`, `2022-04`, … (note: January/April/July/October labels used) |
| `total_folios_crore` | `float64` | No | Total number of investor folios across all scheme categories, in crore units. Includes all open-ended scheme categories. | Positive decimal; e.g. `13.26` = 13.26 crore (132.6 million) folios |
| `equity_folios_crore` | `float64` | No | Number of folios in equity-category schemes (Large Cap, Mid Cap, Small Cap, Flexi Cap, ELSS, Index, etc.), in crore units. | Subset of `total_folios_crore`; typically the largest segment |
| `debt_folios_crore` | `float64` | No | Number of folios in debt-category schemes (Liquid, Gilt, Short Duration, etc.), in crore units. | Subset of `total_folios_crore` |
| `hybrid_folios_crore` | `float64` | No | Number of folios in hybrid-category schemes (Balanced, Aggressive Hybrid, etc.), in crore units. | Subset of `total_folios_crore` |
| `others_folios_crore` | `float64` | No | Number of folios in other categories not classified above (e.g. solution-oriented, fund of funds), in crore units. | Subset of `total_folios_crore`; `equity + debt + hybrid + others ≈ total` |

**Notes:**
- One investor may hold multiple folios (e.g. one per AMC, or separate folios for different schemes within the same AMC)
- Folio count is a proxy for investor reach, not unique investor count

---

## 7. `07_scheme_performance.csv` — Scheme Performance

**Description:** Point-in-time risk and return scorecard for each scheme. Combines trailing returns, benchmark comparison, and risk metrics into a single analytical snapshot. Used for fund comparison and selection analysis.

**Row count:** 40  
**Grain:** One row per scheme (same 40 schemes as fund master)

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `amfi_code` | `int64` | No | Foreign key to `01_fund_master.amfi_code`. | Must match a value in fund master |
| `scheme_name` | `string` | No | Full scheme name. Matches `01_fund_master.scheme_name`. | — |
| `fund_house` | `string` | No | AMC name. Matches `01_fund_master.fund_house`. | — |
| `category` | `string` | No | Scheme sub-category. Matches `01_fund_master.sub_category`. | — |
| `plan` | `string` | No | Plan type (Direct/Regular). Matches `01_fund_master.plan`. | `Direct`, `Regular` |
| `return_1yr_pct` | `float64` | No | Trailing 1-year absolute return of the scheme as a percentage, as of the snapshot date. | Decimal; can be negative |
| `return_3yr_pct` | `float64` | No | Trailing 3-year CAGR (Compound Annual Growth Rate) of the scheme as a percentage. | Decimal; can be negative |
| `return_5yr_pct` | `float64` | No | Trailing 5-year CAGR of the scheme as a percentage. | Decimal; can be negative |
| `benchmark_3yr_pct` | `float64` | No | Trailing 3-year CAGR of the scheme's designated benchmark index, used for active vs passive comparison. | Decimal |
| `alpha` | `float64` | No | Jensen's Alpha — excess annualised return generated by the fund manager relative to the expected return based on the scheme's beta. Positive alpha indicates outperformance. | Decimal; positive = outperformance |
| `beta` | `float64` | No | Measure of the scheme's sensitivity to market movements relative to its benchmark. Beta of 1.0 means the scheme moves in line with the market. | Decimal; typically 0.5–1.5 for equity funds |
| `sharpe_ratio` | `float64` | No | Risk-adjusted return metric: (Scheme return − Risk-free rate) ÷ Standard deviation of scheme returns. Higher is better. | Decimal; positive values are desirable |
| `sortino_ratio` | `float64` | No | Variation of Sharpe ratio that penalises only downside (negative) volatility. More relevant for tail-risk assessment than Sharpe. | Decimal; higher is better |
| `std_dev_ann_pct` | `float64` | No | Annualised standard deviation of daily returns, expressed as a percentage. Measures total volatility of the scheme. | Positive decimal; typically 10–30% for equity funds |
| `max_drawdown_pct` | `float64` | No | Maximum peak-to-trough decline in NAV over the measurement period, as a percentage. Represents the worst-case loss experienced by an investor who bought at the peak. | Negative decimal (e.g. `-21.70` means −21.70% drawdown) |
| `aum_crore` | `int64` | No | Current AUM of the scheme in ₹ crore as of the snapshot date. | Positive integer |
| `expense_ratio_pct` | `float64` | No | Current annual expense ratio of the scheme. May differ from fund master if updated after launch. | Positive decimal |
| `morningstar_rating` | `int64` | No | Morningstar star rating (1–5) assigned to the scheme based on risk-adjusted performance relative to category peers. | Integer: `1`, `2`, `3`, `4`, or `5` |
| `risk_grade` | `string` | No | SEBI riskometer risk label for the scheme. Same taxonomy as `01_fund_master.risk_category`. | `Low`, `Moderate`, `Moderately High`, `High`, `Very High` |

---

## 8. `08_investor_transactions.csv` — Investor Transactions

**Description:** Anonymised individual investor transaction records. Each row represents a single purchase or redemption event. Used for behavioural analysis, geographic distribution, and demographic segmentation of investor activity.

**Row count:** 32,778  
**Grain:** One row per transaction

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `investor_id` | `string` | No | Anonymised unique identifier for the investor. Not traceable to a real individual. | Format: `INVnnnnnn` (e.g. `INV003054`) |
| `transaction_date` | `string (date)` | No | Date on which the transaction was processed. Format: `YYYY-MM-DD`. | Range: `2024-01-01` to `2024-12-31` |
| `amfi_code` | `int64` | No | Foreign key to `01_fund_master.amfi_code`. Identifies the scheme transacted in. | Must match a value in fund master |
| `transaction_type` | `string` | No | Type of transaction executed. | `SIP` (systematic instalment), `Lumpsum` (one-time purchase), `Redemption` (sale/withdrawal) |
| `amount_inr` | `int64` | No | Transaction amount in Indian Rupees (₹). For SIP and Lumpsum, this is the amount invested. For Redemption, this is the amount redeemed. | Positive integer |
| `state` | `string` | No | Indian state from which the transaction originated, based on investor's registered address. | Standard Indian state names (e.g. `Maharashtra`, `Karnataka`, `Telangana`) |
| `city` | `string` | No | City from which the transaction originated. | Standard Indian city names |
| `city_tier` | `string` | No | SEBI/AMFI geographic classification of the investor's city. T30 = Top 30 cities by AUM contribution; B30 = Beyond Top 30 (smaller towns and rural areas). | `T30` (Top 30 cities), `B30` (Beyond Top 30 cities) |
| `age_group` | `string` | No | Age bracket of the investor at the time of transaction. | `18-25`, `26-35`, `36-45`, `46-55`, `56+` |
| `gender` | `string` | No | Self-reported gender of the investor. | `Male`, `Female` |
| `annual_income_lakh` | `float64` | No | Investor's self-declared annual income in lakh Indian Rupees (₹ lakh = ₹100,000). Collected during KYC registration. | Positive decimal; e.g. `7.1` = ₹7.1 lakh per annum |
| `payment_mode` | `string` | No | Payment method used for the transaction. Applicable to purchases (SIP/Lumpsum); Redemptions typically credit back to the investor's bank account. | `UPI`, `Net Banking`, `Mandate` (NACH/auto-debit), `Cheque` |
| `kyc_status` | `string` | No | KYC (Know Your Customer) compliance status of the investor at the time of transaction. Investors must be KYC-verified to invest in mutual funds in India. | `Verified`, `Pending` |

---

## 9. `09_portfolio_holdings.csv` — Portfolio Holdings

**Description:** Stock-level holdings disclosed by each scheme as part of regulatory portfolio disclosure requirements. Each row represents one stock position within one scheme's portfolio. Used for holdings analysis, sector exposure, and overlap detection.

**Row count:** 322  
**Grain:** One row per scheme per stock holding

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `amfi_code` | `int64` | No | Foreign key to `01_fund_master.amfi_code`. Identifies the scheme holding the stock. | Must match a value in fund master |
| `stock_symbol` | `string` | No | Exchange ticker symbol of the stock as listed on NSE (National Stock Exchange) or BSE. | Standard NSE/BSE ticker; e.g. `HDFCBANK`, `POWERGRID`, `GRASIM` |
| `stock_name` | `string` | No | Full legal name of the publicly listed company. | e.g. `HDFC Bank Ltd`, `Power Grid Corporation`, `Grasim Industries Ltd` |
| `sector` | `string` | No | Industry sector classification of the stock. Used for sector-level exposure analysis. | e.g. `Banking`, `Utilities`, `Diversified`, `Technology`, `Consumer Staples` |
| `weight_pct` | `float64` | No | Percentage weight of the stock within the scheme's portfolio, based on market value. All weights for a given `amfi_code` should approximately sum to 100%. | Positive decimal; typically 1–15% per holding |
| `market_value_cr` | `float64` | No | Current market value of the scheme's holding in the stock, in ₹ crore. Calculated as: number of shares held × current market price. | Positive decimal |
| `current_price_inr` | `float64` | No | Current market price of one share of the stock in Indian Rupees (₹). As of the portfolio disclosure date. | Positive decimal |
| `portfolio_date` | `string (date)` | No | Date of the portfolio disclosure. SEBI mandates fund houses to publish full portfolio holdings monthly. Format: `YYYY-MM-DD`. | `2025-12-31` (single snapshot date in this dataset) |

---

## 10. `10_benchmark_indices.csv` — Benchmark Indices

**Description:** Daily closing values of major Indian equity and debt indices used as benchmarks for scheme performance evaluation. Sourced from NSE/BSE and CRISIL index data.

**Row count:** 8,050  
**Grain:** One row per index per trading date

| Column | Data Type | Nullable | Business Definition | Allowed Values / Notes |
|---|---|---|---|---|
| `date` | `string (date)` | No | Trading date for which the index closing value is recorded. Format: `YYYY-MM-DD`. | Range: `2022-01-03` to `2026-05-29`; excludes weekends and holidays |
| `index_name` | `string` | No | Name of the benchmark index. TRI (Total Return Index) variants include dividend reinvestment and are the correct comparator for mutual fund returns. | e.g. `NIFTY50`, `NIFTY100_TRI`, `NIFTY_MIDCAP150_TRI`, `BSE_SMALLCAP_TRI`, `CRISIL_COMPOSITE_BOND` |
| `close_value` | `float64` | No | Official closing value of the index on that trading date. Used to compute index returns over any time period. | Positive decimal; level values vary widely by index |

**Notes:**
- To compute index return between two dates: `((close_value_end / close_value_start) - 1) × 100`
- TRI (Total Return Index) variants are always preferred over Price Return Index for fund performance comparison
- `NIFTY50` represents the price return variant; `NIFTY100_TRI` is the total return benchmark

---

## Entity Relationships

```
01_fund_master (amfi_code PK)
    ├── 02_nav_history        [amfi_code FK]
    ├── 07_scheme_performance [amfi_code FK]
    ├── 08_investor_transactions [amfi_code FK]
    └── 09_portfolio_holdings [amfi_code FK]

03_aum_by_fund_house      [standalone, joined by fund_house name]
04_monthly_sip_inflows    [standalone industry aggregate]
05_category_inflows       [standalone, joined by category name]
06_industry_folio_count   [standalone industry aggregate]
10_benchmark_indices      [standalone, joined by index name to benchmark column in fund master]
```

**Key join paths:**

- **Fund details + Performance:** `01_fund_master` ⟕ `07_scheme_performance` on `amfi_code`
- **Fund details + Transactions:** `01_fund_master` ⟕ `08_investor_transactions` on `amfi_code`
- **Fund details + Holdings:** `01_fund_master` ⟕ `09_portfolio_holdings` on `amfi_code`
- **NAV + Benchmark:** `02_nav_history` joined to `10_benchmark_indices` by date, linking scheme benchmark via `01_fund_master.benchmark`

---

## Glossary

| Term | Definition |
|---|---|
| **AMFI** | Association of Mutual Funds in India — the self-regulatory body for the Indian mutual fund industry. Publishes daily NAV, SIP data, and monthly AUM figures. |
| **AMC** | Asset Management Company — the fund house that creates and manages mutual fund schemes. |
| **AUM** | Assets Under Management — the total market value of investments managed by a fund or AMC. |
| **B30** | Beyond Top 30 — cities outside the top 30 urban centres by AUM contribution. SEBI uses this to track financial inclusion in smaller towns. |
| **CAGR** | Compound Annual Growth Rate — the smoothed annual rate of return over multiple years. |
| **Direct Plan** | A mutual fund plan purchased without a distributor; has a lower expense ratio than a Regular Plan. |
| **ELSS** | Equity Linked Savings Scheme — tax-saving mutual fund with a 3-year lock-in; qualifies for deduction under Section 80C of the Income Tax Act. |
| **Expense Ratio** | Annual fee charged by the AMC as a percentage of AUM, covering management, administration, and distribution costs. Deducted from NAV daily. |
| **Folio** | A unique investor account with a specific AMC. One investor can hold multiple folios across different AMCs. |
| **KYC** | Know Your Customer — mandatory identity and address verification for all mutual fund investors in India, as per SEBI and PMLA regulations. |
| **Lumpsum** | A one-time bulk investment into a mutual fund, as opposed to a systematic instalment. |
| **NAV** | Net Asset Value — the per-unit market value of a mutual fund scheme, calculated daily after market close. |
| **NACH/Mandate** | National Automated Clearing House — an auto-debit mechanism used to collect recurring SIP instalments. |
| **Regular Plan** | A mutual fund plan distributed through intermediaries (agents/distributors); has a higher expense ratio than a Direct Plan to cover distributor commissions. |
| **SEBI** | Securities and Exchange Board of India — the statutory regulator for the securities and mutual fund market in India. |
| **SIP** | Systematic Investment Plan — a facility to invest a fixed amount in a mutual fund at regular intervals (typically monthly). |
| **T30** | Top 30 cities by AUM contribution to the mutual fund industry. SEBI uses this classification for geographic flow analysis. |
| **TER** | Total Expense Ratio — same as Expense Ratio. The all-in annual cost of running the scheme. |
| **TRI** | Total Return Index — an index variant that reinvests dividend income, making it the appropriate benchmark for total return comparisons with mutual funds. |